# 00a · Seudonimización de la base de LEMAS
Convierte el Excel institucional (una hoja por ciclo) en `data/raw/base_seud.csv`:
- deriva `anio_ingreso` desde el código interno;
- reemplaza la **cédula del estudiante** (llave estable entre años, aunque cambie el código por reingreso o cambio de sede) y la cédula del representante por seudónimos HMAC-SHA256;
- deriva el año de ingreso del código interno y luego elimina el código;
- elimina nombres y cédulas.

**⚠️ Privacidad**
- Si usa Colab en la nube, los archivos quedan en servidores de Google durante la sesión. Úselo solo con la autorización escrita del custodio. La alternativa más segura es *Conectar → Conectar a un entorno de ejecución local*.
- **La clave se genera UNA sola vez y se reutiliza siempre.** Con otra clave cambian todos los seudónimos y se pierde el enlace entre años.
- Al terminar, ejecute la última celda para borrar el Excel y la clave de la sesión.

In [ ]:
# @title 0 · Preparar el entorno (ejecutar primero)
# Si el repositorio ya está en GitHub, pegue su URL; si no, se pedirá subir el .zip.
REPO_URL = ""  # @param {type:"string"}

import importlib.util, os, subprocess, sys, zipfile
from pathlib import Path

EN_COLAB = "google.colab" in sys.modules
if EN_COLAB:
    RAIZ = Path("/content/prediccion-matricula-lemas")
    if not RAIZ.exists():
        if REPO_URL:
            subprocess.run(["git", "clone", "-q", REPO_URL, str(RAIZ)], check=True)
        else:
            from google.colab import files
            print("Suba el archivo prediccion-matricula-lemas.zip")
            nombre = next(iter(files.upload()))
            zipfile.ZipFile(nombre).extractall("/content")
else:
    RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

os.chdir(RAIZ)
sys.path.insert(0, str(RAIZ))
REQUERIDOS = {'pandas': 'pandas', 'openpyxl': 'openpyxl'}  # módulo -> paquete pip
faltan = [pip for modulo, pip in REQUERIDOS.items() if importlib.util.find_spec(modulo) is None]
if faltan:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *faltan], check=True)
print("Entorno:", "Google Colab" if EN_COLAB else "local", "| Raíz:", RAIZ)

In [ ]:
# @title 1 · Subir el Excel institucional
ARCHIVO_EXCEL = ""  # @param {type:"string"}
if not ARCHIVO_EXCEL:
    if not EN_COLAB:
        raise ValueError("Indique la ruta del Excel en ARCHIVO_EXCEL")
    from google.colab import files
    ARCHIVO_EXCEL = next(iter(files.upload()))
print("Excel:", ARCHIVO_EXCEL)

In [ ]:
# @title 2 · Clave del custodio
MODO_CLAVE = "subir"  # @param ["subir", "generar"]
sys.path.insert(0, str(RAIZ / "tools"))
import seudonimizar as seud

RUTA_CLAVE = (Path("/content/custodio") if EN_COLAB else RAIZ.parent / "custodio") / "clave_lemas.key"
RUTA_CLAVE.parent.mkdir(parents=True, exist_ok=True)

if MODO_CLAVE == "generar":
    seud.generar_clave(RUTA_CLAVE)
    print("⚠️ Descargue la clave y entréguela al custodio. Sin ella no se puede reidentificar.")
    if EN_COLAB:
        from google.colab import files
        files.download(str(RUTA_CLAVE))
elif not RUTA_CLAVE.exists():
    if not EN_COLAB:
        raise FileNotFoundError(f"Coloque la clave en {RUTA_CLAVE}")
    from google.colab import files
    print("Suba el archivo de clave del custodio (clave_lemas.key)")
    nombre = next(iter(files.upload()))
    Path(nombre).replace(RUTA_CLAVE)
seud.cargar_clave(RUTA_CLAVE)
print("✅ Clave válida")

In [ ]:
# @title 3 · Seudonimizar
COL_ESTUDIANTE = "CI"  # @param {type:"string"}
COL_CODIGO = "Codigo"  # @param {type:"string"}
COL_FAMILIA = "cedulap"  # @param {type:"string"}
# Se eliminan si existen (las que no estén se omiten sin error)
ELIMINAR = ["Codigo", "Orden", "Nombre Completo", "Nombres completos padre de familia",
            "saldo", "deuda", "statusp"]

SALIDA = RAIZ / "data" / "raw" / "base_seud.csv"
SALIDA.parent.mkdir(parents=True, exist_ok=True)
codigo = seud.main(["seudonimizar", "--entrada", ARCHIVO_EXCEL, "--salida", str(SALIDA),
                    "--clave", str(RUTA_CLAVE), "--col-estudiante", COL_ESTUDIANTE,
                    "--col-familia", COL_FAMILIA, "--derivar-anio-ingreso",
                    "--col-codigo", COL_CODIGO, "--eliminar", *ELIMINAR])
assert codigo == 0, "La seudonimización falló: revise el mensaje anterior"

In [ ]:
# @title 4 · Verificar que no queden identificadores
import pandas as pd
resultado = pd.read_csv(SALIDA, dtype=str)
problemas = [c for c in resultado.columns if "nombre" in c.lower()]
for col in resultado.columns:
    muestra = resultado[col].dropna().head(300)
    if len(muestra) and muestra.map(lambda v: seud.validar_cedula_ec(seud.normalizar(v) or "")).mean() > 0.5:
        problemas.append(col)
assert not problemas, f"❌ Aún hay identificadores: {problemas}"
print(f"✅ {len(resultado)} filas, {resultado['hoja'].nunique()} hojas, sin nombres ni cédulas")
print("Columnas:", list(resultado.columns))

In [ ]:
# @title 5 · Descargar resultados (opcional)
DESCARGAR = True  # @param {type:"boolean"}
if EN_COLAB and DESCARGAR:
    from google.colab import files
    files.download(str(SALIDA))
    files.download(str(SALIDA.with_suffix(".acta.json")))

In [ ]:
# @title 6 · Limpieza de la sesión (ejecutar al terminar)
for ruta in (Path(ARCHIVO_EXCEL), RUTA_CLAVE):
    if EN_COLAB and ruta.exists():
        ruta.unlink()
        print("Borrado:", ruta)
print("Listo. base_seud.csv queda en data/raw/ para los notebooks siguientes.")